In [ ]:
import os
from tqdm import tqdm

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

cuda


# Датасет

In [ ]:
DATASET_NAME = "stanfordnlp/imdb"
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
BATCH_SIZE = 16

dataset = load_dataset(DATASET_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
print("Train labels:", torch.unique(tokenized_datasets["train"]["labels"], return_counts=True))
tokenized_datasets.set_format("torch")

train_dataset = tokenized_datasets["train"]
val_dataset = tokenized_datasets["test"]
unlabeled_dataset = tokenized_datasets["unsupervised"]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Unlabeled samples: {len(unlabeled_dataset)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

TypeError: _unique2(): argument 'input' (position 1) must be Tensor, not Column

# Модель

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def train_epoch(loader):
    model.train()
    total_loss = 0
    all_preds =[]
    all_labels =[]

    pbar = tqdm(loader, desc='Training')
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch['input_ids'].size(0)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.append(preds.cpu())
        all_labels.append(batch['labels'].cpu())

        pbar.set_postfix({'Loss': f'{total_loss / len(loader.dataset):.4f}'})

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)

    return total_loss / len(loader.dataset), acc

def validate(loader):
    model.eval()
    total_loss = 0
    all_preds =[]
    all_labels =[]

    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for batch in pbar:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            total_loss += loss.item() * batch['input_ids'].size(0)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.append(preds.cpu())
            all_labels.append(batch['labels'].cpu())

            pbar.set_postfix({'Loss': f'{total_loss / len(loader.dataset):.4f}'})

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)

    return total_loss / len(loader.dataset), acc

In [ ]:
EPOCHS = 3
best_val_acc = 0.0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 50)

    train_loss, train_acc = train_epoch(train_loader)
    val_loss, val_acc = validate(val_loader)

    scheduler.step()

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Saved best model with val accuracy: {val_acc:.4f}")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")


Epoch 1/3
--------------------------------------------------


Validation: 100%|██████████| 1563/1563 [02:48<00:00,  9.26it/s, Loss=0.2288]


Train Loss: 0.2837, Train Acc: 0.8810
Val Loss: 0.2288, Val Acc: 0.9069
Saved best model with val accuracy: 0.9069

Epoch 2/3
--------------------------------------------------


Validation: 100%|██████████| 1563/1563 [02:48<00:00,  9.27it/s, Loss=0.2400]


Train Loss: 0.1606, Train Acc: 0.9392
Val Loss: 0.2400, Val Acc: 0.9009

Epoch 3/3
--------------------------------------------------


Validation: 100%|██████████| 1563/1563 [02:48<00:00,  9.28it/s, Loss=0.3052]


Train Loss: 0.0824, Train Acc: 0.9725
Val Loss: 0.3052, Val Acc: 0.9088
Saved best model with val accuracy: 0.9088

Best validation accuracy: 0.9088


# Предсказания

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

predictions =[]

with torch.no_grad():
    pbar = tqdm(test_loader, desc='Predicting unsupervised data')
    for batch in pbar:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)

        predictions.extend(preds.cpu().numpy())

submission_df = pd.DataFrame({
    'id': range(len(predictions)),
    'text': dataset["unsupervised"]["text"],
    'label_id': predictions
})

LABEL_MAPPING = {0: 'negative', 1: 'positive'}
submission_df['prediction'] = submission_df['label_id'].map(LABEL_MAPPING)

submission_df.to_csv('submission.csv', index=False)
print("\nSubmission saved to submission.csv\n")

pd.set_option('display.max_colwidth', 256)
print(submission_df[['prediction', 'text']].head(15))

Predicting unsupervised data: 100%|██████████| 3125/3125 [05:34<00:00,  9.35it/s]



Submission saved to submission.csv

   prediction  \
0    positive   
1    positive   
2    positive   
3    negative   
4    negative   
5    negative   
6    positive   
7    positive   
8    positive   
9    positive   
10   negative   
11   positive   
12   positive   
13   positive   
14   positive   

                                                                                                                       text  
0   This is just a precious little diamond. The play, the script are excellent. I cant compare this movie with anything ...  
1   When I say this is my favourite film of all time, that comment is not to be taken lightly. I probably watch far too ...  
2   I saw this movie because I am a huge fan of the TV series of the same name starring Roy Dupuis and Pet Wilson. The m...  
3   Being that the only foreign films I usually like star a Japanese person in a rubber suit who crushes little tiny bui...  
4   After seeing Point of No Return (a great movie) and bein